# One intervention: RAG versus all-sessions LLM extraction

Evaluate both methods on the same three-session synthetic transport intervention, using the same local model, question and findings contract. RAG makes one generation per session with earlier retrieved passages and validated memory. Simple LLM sends **all selected spoken conversations in one call**, returning the same timeline and conclusion structure. Production implementation: `src/intervention/simple_llm.py`; the school simulation defaults to this method.

The timeline is retained for output compatibility. Simple LLM timeline entries are retrospective reconstructions: the prompt asks for findings as of each session, but all selected sessions are visible to the model. RAG enforces earlier-session isolation by construction. Neither schema validation nor these checks establishes full factual accuracy or real-world effectiveness.

Findings version 2 separates intended support, direct activities and external requests. Both methods now run one additional semantic review per session. Saved outputs below are historical results from the earlier schema until the notebook is rerun.


In [1]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.intervention.findings import validate_findings
from src.intervention.ollama_client import OllamaClient
from src.intervention.rag import InterventionRAG
from src.intervention.reports import write_report
from src.intervention.session_index import SessionIndex
from src.intervention.simple_llm import SimpleLLM
from src.intervention.transcripts import load_sessions
from src.utils.constants import HISTORIES

MODEL = os.getenv("EVALUATION_MODEL", "qwen3.5:latest")
HISTORY_ID = "01_transport_assessment/intervention_student_003"
THROUGH_SESSION = 3
QUESTION = "What barriers remain, what happened to agreed actions, and is there evidence of delivered support or improvement?"
OUTPUT_DIR = PROJECT_ROOT / "outputs/intervention_extraction_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sessions = load_sessions(HISTORIES)
selected = [s for s in sessions if s.history_id == HISTORY_ID and s.number <= THROUGH_SESSION]
print(f"Model: {MODEL}; history: {HISTORY_ID}; recorded sessions: {len(selected)}")
for session in selected:
    print(f"Session {session.number}: {len(session.turns)} spoken turns")

Model: qwen3.5:latest; history: 01_transport_assessment/intervention_student_003; recorded sessions: 3
Session 1: 114 spoken turns
Session 2: 64 spoken turns
Session 3: 60 spoken turns


## Transcript-based reference, fixed before generation

Both methods receive spoken turns and allowlisted descriptive metadata (intervention type, date, stage). Prewritten outcome notes are excluded. These reference excerpts are for evaluation and are **not passed as model instructions or labels**. The final session states that the difficulty remains on some days. The earlier action (record two journey times) was reported done in session 2; the next action (check an alternative departure) remains unclear in session 3. External delivery and school-attendance improvement are not established.

The three automated checks below cover conservative final indicators. A barrier marked `no` or `partial` is accepted because the student reports persistence with some earlier change. Delivery is recorded per intended component, direct activity and external request; it is excluded from this narrow automatic rubric. Runtime source checks and semantic reviews check action tracking, delivery scope and narrative consistency. Inspect the source evidence when assessing accuracy.

In [2]:
REFERENCE_QUOTES = {
    "initial_barrier": (1, "The route involves a crossing that makes me nervous."),
    "first_action_done": (2, "Yes, I completed the specific step we agreed and made a note of what happened."),
    "second_action_uncertain": (3, "I am not sure whether I did it exactly as planned."),
    "barrier_persists": (3, "The difficulty is still present on some days."),
    "outside_support_unproven": (3, "We talked and practiced, but extra help has not automatically been delivered."),
}
for label, (number, quote) in REFERENCE_QUOTES.items():
    assert quote in next(s.text for s in selected if s.number == number), label
    print(f"{label} [session {number}]: {quote}")

REFERENCE = {
    "primary_barrier_resolved": ["no", "partial"],
    "attendance_improved": ["unknown"],
    "intervention_helpful": ["unknown"],
}

initial_barrier [session 1]: The route involves a crossing that makes me nervous.
first_action_done [session 2]: Yes, I completed the specific step we agreed and made a note of what happened.
second_action_uncertain [session 3]: I am not sure whether I did it exactly as planned.
barrier_persists [session 3]: The difficulty is still present on some days.
outside_support_unproven [session 3]: We talked and practiced, but extra help has not automatically been delivered.


## Run both implementations

Requires Ollama with the configured model already installed. There are no retries. Reports are saved immediately after each method succeeds; failures are retained in the comparison and cannot receive a passing score. Timing includes model calls and Python analysis, and is descriptive: RAG runs first, so cold-start/cache effects are not controlled. Prompt size is measured in characters including the schema, not tokens. The metered client below is notebook-only instrumentation.

In [3]:
class MeteredClient:
    def __init__(self):
        self.client = OllamaClient(model=MODEL, timeout=600)
        self.calls = []
        self.requests = []

    def generate(self, messages, schema):
        self.requests.append({"messages": messages, "schema": schema})
        started = perf_counter()
        measurement = {"prompt_chars": len(json.dumps(messages)) + len(json.dumps(schema))}
        self.calls.append(measurement)
        try:
            return self.client.generate(messages, schema)
        finally:
            measurement["elapsed_seconds"] = round(perf_counter() - started, 3)

reports = {}
measurements = {}
for method in ("rag", "simple"):
    client = MeteredClient()
    index = None
    started = perf_counter()
    error = None
    try:
        if method == "rag":
            index = SessionIndex()
            index.build(selected)
            analyser = InterventionRAG(index, client)
        else:
            analyser = SimpleLLM(selected, client)
        report = analyser.analyse(HISTORY_ID, THROUGH_SESSION, QUESTION)
        for step in report["timeline"]:
            validate_findings(step["findings"])
        assert [step["session_number"] for step in report["timeline"]] == [s.number for s in selected]
        assert report["conclusion"] == report["timeline"][-1]["findings"]
        reports[method] = report
        write_report(OUTPUT_DIR / method / "conclusion.json", report)
    except (ValueError, RuntimeError, AssertionError) as exc:
        error = str(exc)
    finally:
        write_report(OUTPUT_DIR / method / "requests.json", client.requests)
        if index is not None:
            index.close()
        measurements[method] = {
            "status": "failed" if error else "completed", "error": error,
            "model_calls": len(client.calls),
            "elapsed_seconds": round(perf_counter() - started, 3),
            "total_prompt_chars": sum(call["prompt_chars"] for call in client.calls),
            "calls": client.calls,
        }
        print(method, json.dumps(measurements[method]), flush=True)

rag {"status": "completed", "error": null, "model_calls": 3, "elapsed_seconds": 127.358, "total_prompt_chars": 75852, "calls": [{"prompt_chars": 24883, "elapsed_seconds": 44.297}, {"prompt_chars": 23613, "elapsed_seconds": 38.281}, {"prompt_chars": 27356, "elapsed_seconds": 44.726}]}
simple {"status": "completed", "error": null, "model_calls": 1, "elapsed_seconds": 84.705, "total_prompt_chars": 39530, "calls": [{"prompt_chars": 39530, "elapsed_seconds": 84.677}]}


## Compare structure, outcomes and the fixed reference

Agreement is not accuracy: two methods can agree on an unsupported claim. The reference checks are a small, explicitly bounded rubric for this one synthetic intervention. The saved reports support review of richer findings and action progression.

In [4]:
rows = []
for method, measurement in measurements.items():
    row = {"method": method, **{key: measurement[key] for key in
           ("status", "model_calls", "elapsed_seconds", "total_prompt_chars")}}
    if method in reports:
        findings = reports[method]["conclusion"]
        values = {**findings["outcome"],
                  "intended_status": findings["intervention"]["intended_intervention"]["status"],
                  "direct_activity_statuses": {c["component_id"]: c["status"]
                                               for c in findings["intervention"]["direct_activities"]},
                  "external_support_statuses": {c["component_id"]: c["status"]
                                                for c in findings["intervention"]["external_support"]}}
        checks = {key: values[key] in allowed for key, allowed in REFERENCE.items()}
        row.update(values)
        row.update({"reference_checks_passed": sum(checks.values()), "reference_checks_total": len(checks),
                    "reference_checks": checks})
    else:
        row.update({"reference_checks_passed": 0, "reference_checks_total": len(REFERENCE),
                    "error": measurement["error"]})
    rows.append(row)

agreement = None
if all(method in reports for method in ("rag", "simple")):
    agreement = {key: reports["rag"]["conclusion"]["outcome"][key] == reports["simple"]["conclusion"]["outcome"][key]
                 for key in reports["rag"]["conclusion"]["outcome"]}
    assert measurements["rag"]["model_calls"] == 2 * len(selected)
    assert measurements["simple"]["model_calls"] == 1 + len(selected)

comparison = {
    "history_id": HISTORY_ID, "as_of_session": THROUGH_SESSION, "model": MODEL,
    "question": QUESTION, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(), "num_ctx": 32768, "temperature": 0,
    "transcript_sha256": hashlib.sha256(json.dumps([
        {"number": s.number, "turns": s.turns, "metadata": s.metadata} for s in selected], ensure_ascii=False).encode()).hexdigest(),
    "reference": REFERENCE, "reference_quotes": REFERENCE_QUOTES,
    "results": rows, "outcome_agreement": agreement, "measurements": measurements,
    "limitations": ["One synthetic intervention; no general accuracy or causal claim.",
                    "Three reference checks do not measure full narrative accuracy.",
                    "Single sequential run: cold-start and caching effects are uncontrolled.",
                    "Simple timeline is reconstructed with all selected sessions visible.",
                    "Source checks and separate semantic reviews remain fallible, not independent verification."],
}
write_report(OUTPUT_DIR / "comparison.json", comparison)
import pandas as pd
from IPython.display import display
display(pd.DataFrame(rows).drop(columns=["reference_checks"], errors="ignore"))
print("Final outcome agreement:", agreement)
print("Saved to", OUTPUT_DIR.relative_to(PROJECT_ROOT))

,method,status,model_calls,elapsed_seconds,total_prompt_chars,primary_barrier_resolved,attendance_improved,intervention_helpful,service_delivered,reference_checks_passed,reference_checks_total
0,rag,completed,3,127.358,75852,no,unknown,unknown,False,4,4
1,simple,completed,1,84.705,39530,no,unknown,unknown,False,4,4


Final outcome agreement: {'primary_barrier_resolved': True, 'attendance_improved': True, 'intervention_helpful': True}
Saved to outputs/intervention_extraction_evaluation


## Review the intervention progression

Check that session 2 retains the reported completion of recording journey times, while session 3 leaves completion of checking the alternative departure uncertain. An activity completed in the mentoring room must not establish outside delivery or resolution of the unsafe crossing. Compare the original turns above and source transcripts with both narratives.

In [5]:
for method, report in reports.items():
    print(f"\n--- {method.upper()} ---")
    for step in report["timeline"]:
        finding = step["findings"]
        print(f"Session {step['session_number']}: {finding['session_summary']}")
        print("Actions:", json.dumps(finding["actions"], ensure_ascii=False))
        print("Tracked details:", json.dumps(finding["tracked_details"], ensure_ascii=False))
    print("Final next steps:", report["conclusion"]["next_steps"])
    print("Final uncertainties:", report["conclusion"]["uncertainties"])
    print("Final conclusion:", report["conclusion"]["conclusion"])

# A failed live method must not look like a successful completed evaluation.
assert all(m["status"] == "completed" for m in measurements.values()), measurements


--- RAG ---
Session 1: Initial session focused on establishing a safe, practical approach to a transport barrier (nervousness about a specific crossing). The mentor and student identified the main difficulty as uncertainty regarding an approved alternative route. A small, manageable activity was attempted in-session: mapping the usual route and identifying difficult connection points. The student agreed to a between-session task of recording two actual journey times before the next meeting.
Tracked details: [{"detail": "Between-session task: Record two actual journey times before the next meeting.", "status": "pending"}, {"detail": "External support request: Verification of an approved alternative route or transport entitlement.", "status": "pending"}]
Session 2: Follow-up session focused on reviewing a completed between-session task (recording journey times) and comparing data with the timetable. The student confirmed completion of the specific step but reported that the wider barrie

## Review of the updated run

RAG: **3 calls, 129.409 seconds, 61,702 prompt characters**. SimpleLLM: **1 call, 75.261 seconds, 35,088 prompt characters**. Both passed **3/4 unchanged final-field checks**, compared with 4/4 in the prior run. Both incorrectly set intervention helpfulness to yes while leaving delivery unknown. SimpleLLM retained the two-journey-times action and recognized session helpfulness, but still weakened the final crossing concern.

SimpleLLM preserved the central action sequence: pending journey recording in session 1, reported completion and a new departure task in session 2, and unknown departure-task completion in session 3. This is a bounded observation, not proof of temporal isolation.

Exact model requests are saved alongside each report in `requests.json`. See [the detailed saved evaluation](../outputs/intervention_extraction_evaluation/evaluation.md) for narrative review, environment limitations and comparison with the prior prompts. These measurements describe this saved run; re-review after rerunning.
